# Model selection and operational interpretation

## Objective
Read actual selection evidence and independently recompute official endpoint metrics.

## Method
Use the official FD001 files and saved engine-separated artifacts. Rebuild training with `python scripts/train_models.py` before rerunning if configuration changes.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'app.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from src.data_loader import read_raw
from src.features import make_features, targets
train = read_raw()
test = read_raw('test')
results = json.loads((ROOT / 'artifacts/metrics/results.json').read_text())


In [2]:
from src.evaluation import regression_metrics
fleet=pd.read_csv(ROOT/'data/processed/fleet.csv')
print(pd.read_csv(ROOT/'artifacts/metrics/comparison.csv').to_string(index=False))
print('Selected:',results['selected'])
print('Uncapped test:',regression_metrics(fleet.actual_rul,fleet.rul))
print('Interval:',results['interval'])
print('Classification:',results['classification'])
import plotly.express as px
fig=px.scatter(fleet,x='actual_rul',y='rul',color='risk',hover_name='engine',title='Official endpoint prediction versus truth')
fig.update_layout(template='plotly_white')

            model       MAE      RMSE        R2   NASA_score  fit_seconds
  Median baseline 46.800000 58.581567 -0.018048 38874.696267     0.004918
            Ridge 29.172919 43.181054  0.446863 35942.059050     0.035866
    Random forest 26.814624 38.078171  0.569871  3201.901171    14.151263
Gradient boosting 28.075778 38.925851  0.550507  3750.009089     0.639345
Selected: Random forest
Uncapped test: {'MAE': 14.125933898341362, 'RMSE': 18.85868891305181, 'R2': 0.7940492162701438, 'NASA_score': 648.2540381987831}
Interval: {'nominal_coverage': 0.8, 'test_coverage': 0.97, 'radius': 41.71694913642378, 'calibration_engines': 20}
Classification: {'precision': 0.875, 'recall': 0.84, 'F1': 0.8571428571428571, 'ROC_AUC': 0.9823999999999999, 'PR_AUC': 0.935720453395267, 'Brier': 0.0509605308207314, 'confusion_matrix': [[72, 3], [4, 21]]}


Interactive Plotly figure

## Interpretation and conclusion
Test results are reported only after model selection. Wide intervals explain high observed coverage; neither test accuracy nor coverage establishes real aircraft safety.